# Chapter 3 (hadronic) — Notebook 4: Fit m_top

**Goals**

- Fit m(jjb) with Gaussian + polynomial and extract m_top ± σ_stat.
- Compare with Group L.

In [ ]:
%matplotlib inline
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt

from topmass import io, kinematics, selection, plotting, fitting, neutrino, pairing, weights, style
from topmass.constants import M_W, M_TOP

In [ ]:
io.setup()                                  # select release 2025e-13tev-beta
samples = io.build_samples()                # skim '3J1LMET30', https
events = io.load_process('ttbar', samples, fraction=0.3)
print('Number of events:', len(events))

In [ ]:
cuts = selection.SemilepCuts(n_jets_min=4, n_bjets_min=2)
events = events[selection.semilep_preselection(events, cuts)]

lep = kinematics.leading_lepton(events)
jets = kinematics.jet_vectors(events)
is_b = events.jet_btag_quantile >= cuts.btag_quantile_min
b_jets, light = jets[is_b], jets[~is_b]
keep = (ak.num(b_jets) >= 2) & (ak.num(light) >= 2)
lep, b_jets, light = lep[keep], b_jets[keep][:, :2], light[keep]
_, b_had = pairing.assign_bjets(b_jets, lep)
j1, j2, _ = pairing.best_W_pair(light)
m_top = ak.to_numpy((j1 + j2 + b_had).mass)

counts, edges = np.histogram(m_top, bins=40, range=(120, 280))
result = fitting.fit_topmass(counts.astype(float), edges)
print(f'm_top (hadronic) = {result.params["mu"]:.2f} ± {result.errors["mu"]:.2f} GeV')

centres = 0.5 * (edges[:-1] + edges[1:])
plt.errorbar(centres, counts, yerr=np.sqrt(np.maximum(counts, 1)), fmt='o', markersize=3, label='reco')
xfit = np.linspace(edges[0], edges[-1], 400)
plt.plot(xfit, fitting.signal_plus_bkg(xfit, **{k: result.params[k] for k in ('n_sig','mu','sigma','c0','c1','c2')}), label='fit')
plt.axvline(172.5, color='grey', ls='--', label='generator $m_t$')
plt.xlabel(r'$m(jjb)$ [GeV]'); plt.legend()

## ✏️ Your turn 4.1 — closure

▶️ Change the match cut and re-run.

There is no truth top-mass branch, so we validate differently: keep only events whose two light jets
**and** hadronic b-jet each ΔR-match a generator-level `truth_jet` within `DR_MATCH`, then re-fit. On
this clean, correctly-paired subset the fitted m_top should sit very close to 172.5 GeV.

> **Stretch (optional):** loosen `DR_MATCH` toward 0.6 — more events enter but with more wrong
> pairings, and the peak should drift.

In [ ]:
DR_MATCH = 0.4    # ✏️ try 0.2, 0.4, 0.6

events_k = events[keep]                       # the events behind m_top above
truth = kinematics.truth_jet_vectors(events_k)

def is_matched(obj):
    dr = ak.fill_none(ak.min(obj.deltaR(truth), axis=1), 99.0)
    return ak.to_numpy(dr) < DR_MATCH

good = is_matched(j1) & is_matched(j2) & is_matched(b_had)
print(f'{int(good.sum())} / {len(good)} events fully truth-matched (ΔR < {DR_MATCH})')

counts, edges = np.histogram(m_top[good], bins=40, range=(120, 280))
result = fitting.fit_topmass(counts.astype(float), edges)
print(f'm_top (truth-matched) = {result.params["mu"]:.2f} ± {result.errors["mu"]:.2f} GeV   (target 172.5)')

## Wrap-up: report

1. Quote m_top ± stat.
2. Discuss systematics (jet energy scale, combinatorial background from wrong pairing).
3. Compare with Group L's leptonic-side result and discuss any tension.